In [1]:
# 自动检查并安装依赖项
import importlib
import subprocess
import sys

def check_and_install(package):
    try:
        importlib.import_module(package)
        print(f"{package} 已安装")
    except ImportError:
        print(f"{package} 未安装，正在安装...")
        # 使用Popen代替check_call以支持实时输出
        process = subprocess.Popen(
            [sys.executable, "-m", "pip", "install", package],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        # 实时读取并打印输出
        for line in iter(process.stdout.readline, ''):
            print(line.strip())
        
        process.stdout.close()
        process.wait()
        
        if process.returncode == 0:
            print(f"{package} 安装完成")
        else:
            print(f"{package} 安装失败，返回代码 {process.returncode}")

# 检查并安装必要的依赖项
required_packages = ['pandas', 'matplotlib', 'seaborn']
for package in required_packages:
    check_and_install(package)

In [1]:
import re
import json
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Path to log file
log_file_path = '/root/autodl-fs/mimibot-factory/grpo_log_0322.txt'

# Read the log file
with open(log_file_path, 'r') as file:
    log_content = file.read()

log_list = []
# 逐行扫描
for line in log_content.split('\n'):
    # 匹配日志行的正则表达式
    """
    {'loss': 0.0254, 'grad_norm': 17.50685691833496, 'learning_rate': 9.992292139465165e-05, 'rewards/reward_format': 22.70833384990692, 'rewards/reward_no_repetition': 0.0, 'rewards/reward_similarity': 752.6263809204102, 'reward': 775.3347015380859, 'reward_std': 345.1836128234863, 'completion_length': 73.27083778381348, 'kl': 0.6342208124697208, 'epoch': 0.18}
    """
    match = re.search(r"{'loss': (.*?)}", line)
    if match:
        match_line = match.group(0).replace('\'', '\"')
        # 加载为字典
        log_dict = json.loads(match_line)
        log_list.append(log_dict)
print(log_list)

In [10]:
# 从log_list里面提取数据，用matplotlib绘图
reword_list = []
epoch_list = []
reword_format_list = []
reword_repetition_list = []
reword_similarity_list = []
for log in log_list:
    reword_list.append(log['reward'])
    epoch_list.append(log['epoch'])
    reword_format_list.append(log['rewards/reward_format'])
    reword_repetition_list.append(log['rewards/reward_no_repetition'])
    reword_similarity_list.append(log['rewards/reward_similarity'])

# MATPLOTLIB 绘图
plt.plot(reword_list, label='reward')
plt.plot(reword_format_list, label='reward_format')
plt.plot(reword_repetition_list, label='reward_no_repetition')
plt.plot(reword_similarity_list , label='reward_similarity')
plt.legend()
plt.xlabel('step')
plt.ylabel('reward')
plt.title('reward curve')


